# 🎓 CURSO PRÁCTICO SÚPER DETALLADO: ARQUITECTURA MODEL CONTEXT PROTOCOL (MCP)

## ¿Qué es y cómo funciona el Model Context Protocol (MCP)?

Model Context Protocol (MCP) es un protocolo estandarizado de código abierto que actúa como un puente de comunicación unificado entre los **Modelos de Inteligencia Artificial (LLMs)** y los **sistemas o bases de datos locales**. 

Su principal fortaleza radica en la **desacoplación**:
1. **Servidor MCP (`mcp_server.py`)**: Contiene todo el código pesado en Python para conectarse a MySQL, limpiar nulos e imputar datos. Este servidor "registra" y ofrece estas funciones como "Herramientas" (Tools).
2. **Cliente MCP (Este Notebook)**: Lanza el servidor en el fondo, lee sus herramientas disponibles, le expone estas herramientas al LLM en la nube y ejecuta las órdenes físicas dictadas por el modelo sobre tu máquina local.

En esta guía dividida celda por celda aprenderás de forma súper granular cómo funciona esta comunicación.

---

## 🛠️ Paso 1: Instalación e Importación de Librerías
En esta celda importamos las herramientas clave para nuestro orquestador.
- `mcp`: Permite crear sesiones de comunicación estructuradas.
- `openai`: El SDK estándar compatible con OpenRouter para la consulta al LLM.
- `dotenv`: Para cargar de manera segura tus variables de entorno privadas sin hardcodearlas.

In [ ]:
import os
import asyncio
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

---

## 🔑 Paso 2: Carga Segura de Credenciales (.env)
Cargamos las variables privadas del archivo local `.env` (que está protegido de Git mediante `.gitignore`). Esto asegura que tus contraseñas y llaves de API nunca se fuguen a repositorios públicos.

In [ ]:
# Cargar variables del .env
load_dotenv()

# Validación amigable del entorno
if not os.getenv("OPENROUTER_API_KEY"):
    print("⚠️ ADVERTENCIA: La variable de entorno OPENROUTER_API_KEY no está configurada.")
else:
    print("✅ Credenciales y configuraciones de entorno cargadas con éxito.")

---

## ⚙️ Paso 3: Configurar los Parámetros del Servidor MCP
Definimos el puente de conexión. Le decimos a nuestro cliente que levante en segundo plano el script `mcp_server.py` utilizando el comando reproducible `uv run python`.

In [ ]:
server_params = StdioServerParameters(
    command="uv",
    args=["run", "python", "mcp_server.py"],
    env=os.environ.copy() # Pasamos las credenciales seguras al servidor hijo
)
print("✅ Parámetros de comunicación del servidor listos.")

---

## 💬 Paso 4: Definición de la Consulta / Prompt
Aquí puedes elegir qué instrucción enviarle al modelo de Inteligencia Artificial.
1. **Predeterminado**: Extrae 100 registros y ejecuta la limpieza.
2. **Personalizado**: Puedes escribir cualquier consulta alternativa en la variable `prompt_personalizado` (por ejemplo, *"Solo extrae 10 registros de la base de datos MySQL"* o *"Ejecuta la limpieza sobre el archivo local"*) y el Notebook la priorizará automáticamente.

In [ ]:
# --- CONSULTA PREDETERMINADA ---
prompt_predeterminado = (
    "Por favor, extrae 100 transacciones de la base de datos MySQL "
    "utilizando tus herramientas locales y luego procesa los datos "
    "para dejarlos listos para entrenar un modelo de ML."
)

# --- CONSULTA PERSONALIZADA (Escribe tu propia instrucción aquí si deseas probar otra cosa) ---
prompt_personalizado = ""

# Selección dinámica del prompt activo
prompt_activo = prompt_personalizado if prompt_personalizado.strip() else prompt_predeterminado

print("📋 Prompt seleccionado y configurado para la inferencia del modelo:")
print("-" * 80)
print(prompt_activo)
print("-" * 80)

---

## 🧠 Paso 5: Inicialización del Cliente de Inferencia (OpenRouter)
Instanciamos nuestro cliente de OpenRouter para conectarnos con el modelo Nvidia Nemotron.

In [ ]:
llm_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY", "")
)
print("✅ Cliente API de OpenRouter inicializado exitosamente.")

---

## 🤝 Paso 6: Handshake y Descubrimiento Dinámico de Herramientas MCP
En esta celda abrimos una conexión rápida al Servidor MCP local para descubrir qué herramientas expone. Traducimos sus firmas al formato JSON Schema que exige el modelo LLM para entender qué funciones puede invocar.

In [ ]:
async def descubrir_herramientas_mcp():
    print("🔌 Contactando al Servidor MCP local...")
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # Solicitar catálogo al servidor local
            tools_response = await session.list_tools()
            
            # Traducir a formato compatible con OpenAI
            openrouter_tools = []
            for tool in tools_response.tools:
                openrouter_tools.append({
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema
                    }
                })
            return openrouter_tools, [t.name for t in tools_response.tools]

# Ejecutamos el descubrimiento asíncrono
openrouter_tools, herramientas_disponibles = await descubrir_herramientas_mcp()
print(f"✅ Handshake completo. Herramientas MCP registradas para el LLM: {herramientas_disponibles}")

---

## 🚀 Paso 7: Ejecución del Flujo y Orquestación en Vivo
Definimos la función para ejecutar todo el pipeline de inferencia y ejecución. 
1. Envía el prompt y el catálogo de herramientas al LLM en la nube.
2. El LLM decide qué herramientas llamar.
3. El Notebook intercepta la respuesta y ejecuta las funciones locales reales de `mcp_server.py` sobre tu MySQL.
4. Imprime los resultados obtenidos paso a paso en pantalla.

In [ ]:
async def correr_pipeline_mcp(prompt):
    print(f"🧠 Enviando consulta a nvidia/nemotron-3-super-120b-a12b:free...")
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            messages = [
                {
                    "role": "system", 
                    "content": "Eres un agente científico de datos altamente técnico. Tu tarea es extraer la data de MySQL y transformarla usando tus herramientas. Responde de forma muy concisa."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ]
            
            # Consulta al LLM
            response = await llm_client.chat.completions.create(
                model="nvidia/nemotron-3-super-120b-a12b:free",
                messages=messages,
                tools=openrouter_tools
            )
            
            mensaje_modelo = response.choices[0].message
            
            # Procesador dinámico de tool calls
            if mensaje_modelo.tool_calls:
                print("\n🤖 [El modelo determinó llamar herramientas locales en tu máquina]:")
                for tool_call in mensaje_modelo.tool_calls:
                    nombre_funcion = tool_call.function.name
                    argumentos = json.loads(tool_call.function.arguments)
                    
                    print(f"   ➔ ⚙️ Ejecutando herramienta local: {nombre_funcion}() con argumentos {argumentos}")
                    
                    # Ejecutamos el código real de mcp_server.py
                    resultado = await session.call_tool(nombre_funcion, argumentos)
                    
                    # Mostramos el reporte
                    print(f"   ✅ [Servidor Local]: {resultado.content[0].text}\n")
                print("🎉 Pipeline MCP orquestado con éxito.")
            else:
                print("\n💬 Respuesta del modelo (sin llamadas a herramientas):")
                print(mensaje_modelo.content)

print("✅ Orquestador asíncrono configurado y listo.")

---

## 🚀 Paso 8: Ejecución en Vivo
Corremos la ejecución del pipeline con el prompt seleccionado. Puedes ver las respuestas e interacciones en tiempo real.

In [ ]:
await correr_pipeline_mcp(prompt_activo)